## tl;dr

같은 CCTV형 입력과 같은 속성 필드 계약으로 측정한 후보 비교다. 현재 측정 후보 중 Qwen3-VL-2B가 속성 정확도는 가장 높았지만 image/track exact match가 0%이므로 실종자 식별용 최종 모델로 승인할 수 없다. 서버용 Qwen3-VL-8B와 대형 대체 VLM은 원격 인증 후 같은 하네스로 추가 측정해야 한다.

## Context & Methods

목표는 CCTV 프레임에서 성별, 연령대, 머리색, 시점, 상의·하의 색/종류, 액세서리 속성을 비교하는 것이다. Simuletic CCTV-Pedestrian-1K의 synthetic high-angle CCTV 이미지와 PA-100K의 실제 surveillance crop을 proxy로 사용했다. Qwen3-VL-2B는 JSON 생성, CLIP ViT-B/32는 field별 zero-shot 후보 점수, Florence-2-large-ft는 상세 캡션 후 정규화로 평가했다. 지표는 attribute accuracy, image exact match, track exact match, 구조화 출력률, p50/p95 지연이다. 데이터셋 라벨이 프로젝트의 모든 색상·질감·신원 속성을 포함하지 않으므로 proxy 결과이며 배포 정확도를 의미하지 않는다.

In [1]:
from pathlib import Path
import json

root = Path.cwd()
result_paths = [
    root / 'experiments/results/qwen3-vl-2b.json',
    root / 'experiments/results/clip-vit-b32.json',
    root / 'experiments/results/florence-2-large.json',
    root / 'experiments/results/florence-2-large-ft.json',
]
records = []
for path in result_paths:
    with path.open(encoding='utf-8') as handle:
        records.append(json.load(handle))
fields = ['gender', 'age', 'hair_color', 'viewpoint', 'top_color', 'top_type', 'bottom_color', 'bottom_type', 'accessory']
records

[{'status': 'valid',
  'hardware': 'NVIDIA GeForce RTX 4050 Laptop GPU',
  'torch': '2.11.0+cu128',
  'runtime': {'python': '3.14.6',
   'transformers': '5.14.1',
   'torchvision': '0.26.0+cu128',
   'cuda': '12.8'},
  'model': 'qwen3-vl-2b',
  'checkpoint': 'experiments/models/Qwen3-VL-2B-Instruct',
  'dataset_counts': {'simuletic': 15, 'pa100k': 30},
  'summary': {'model': 'qwen3-vl-2b',
   'examples': 45,
   'groups': 33,
   'attribute_accuracy': 0.3929824561403509,
   'image_exact_match': 0.0,
   'track_exact_match': 0.0,
   'track_groups_evaluated': 3,
   'structured_valid_rate': 0.5555555555555556,
   'latency_s_p50': 4.914235299977008,
   'latency_s_p95': 9.845891600009054,
   'field_accuracy': {'gender': 0.37777777777777777,
    'age': 0.2,
    'hair_color': 0.6,
    'viewpoint': 0.4,
    'top_color': 0.3333333333333333,
    'top_type': 0.6666666666666666,
    'bottom_color': 1.0,
    'bottom_type': 0.26666666666666666,
    'accessory': 0.37777777777777777},
   'status': 'valid

## Data

| 데이터 | 용도 | 한계 |
|---|---|---|
| Simuletic CCTV-Pedestrian-1K | 고각도 CCTV형 person attribute proxy | synthetic, license 문구 확인 필요 |
| PA-100K | 실제 surveillance crop proxy | 색상·질감·신원 라벨 제한 |

중복 파일명은 누수와 라벨 불일치를 막기 위해 제외했다. 유효 샘플은 Simuletic 15장, PA-100K 30장으로 총 45장이다.

In [2]:
from statistics import median

def canonical(field, value):
    text = value.lower().strip().replace("_", " ") if isinstance(value, str) else ""
    if field == "gender":
        if any(token in text for token in ("woman", "female", "girl")):
            return "female"
        if any(token in text for token in ("man", "male", "boy")):
            return "male"
    if field == "viewpoint":
        if "front" in text:
            return "front"
        if "back" in text:
            return "back"
        if "side" in text:
            return "side"
    if field == "accessory":
        if text in {"", "none", "no accessory", "nothing"}:
            return "none"
        for token in ("backpack", "handbag", "umbrella", "sunglasses", "smartphone", "cap"):
            if token in text:
                return token
    return " ".join(text.split())

def per_dataset_rows(record):
    grouped = {}
    for prediction in record["predictions"]:
        if prediction.get("error") is not None:
            continue
        dataset = prediction["dataset"]
        bucket = grouped.setdefault(dataset, {"correct": 0, "total": 0, "valid": 0, "latencies": []})
        target = prediction.get("target") or {}
        predicted = prediction.get("predicted") or {}
        for field in fields:
            expected = target.get(field)
            if expected in (None, "", "unknown"):
                continue
            actual = predicted.get(field)
            bucket["total"] += 1
            bucket["correct"] += int(bool(actual) and canonical(field, actual) == expected)
        bucket["valid"] += int(prediction.get("structured_valid", False))
        bucket["latencies"].append(prediction["latency_s"])
    rows = []
    for dataset, bucket in grouped.items():
        rows.append({
            "model": record["model"],
            "dataset": dataset,
            "examples": len(bucket["latencies"]),
            "attribute_accuracy_pct": round(100 * bucket["correct"] / bucket["total"], 1),
            "structured_valid_pct": round(100 * bucket["valid"] / len(bucket["latencies"]), 1),
            "latency_p50_s": round(median(bucket["latencies"]), 3),
        })
    return rows

per_dataset = [row for record in records for row in per_dataset_rows(record)]
per_dataset

[{'model': 'qwen3-vl-2b',
  'dataset': 'simuletic',
  'examples': 15,
  'attribute_accuracy_pct': 71.9,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 4.93},
 {'model': 'qwen3-vl-2b',
  'dataset': 'pa100k',
  'examples': 30,
  'attribute_accuracy_pct': 10.0,
  'structured_valid_pct': 33.3,
  'latency_p50_s': 4.657},
 {'model': 'clip-vit-b32',
  'dataset': 'simuletic',
  'examples': 15,
  'attribute_accuracy_pct': 40.0,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 0.902},
 {'model': 'clip-vit-b32',
  'dataset': 'pa100k',
  'examples': 30,
  'attribute_accuracy_pct': 37.3,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 0.655},
 {'model': 'florence-2-large',
  'dataset': 'simuletic',
  'examples': 15,
  'attribute_accuracy_pct': 34.8,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 2.046},
 {'model': 'florence-2-large',
  'dataset': 'pa100k',
  'examples': 30,
  'attribute_accuracy_pct': 31.3,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 1.766},
 {'model': 'flo

## Results

전체 요약과 데이터셋별 결과를 함께 확인한다. structured_valid_rate는 모델별 출력 경로가 다르므로 정확도와 같은 의미의 점수가 아니다.

In [3]:
summary = []
for record in records:
    item = record['summary']
    summary.append({
        'model': item['model'],
        'examples': item['examples'],
        'attribute_accuracy_pct': round(100 * item['attribute_accuracy'], 1),
        'image_exact_pct': round(100 * item['image_exact_match'], 1),
        'track_exact_pct': round(100 * item['track_exact_match'], 1),
        'structured_valid_pct': round(100 * item['structured_valid_rate'], 1),
        'latency_p50_s': round(item['latency_s_p50'], 3),
        'latency_p95_s': round(item['latency_s_p95'], 3),
    })
summary

[{'model': 'qwen3-vl-2b',
  'examples': 45,
  'attribute_accuracy_pct': 39.3,
  'image_exact_pct': 0.0,
  'track_exact_pct': 0.0,
  'structured_valid_pct': 55.6,
  'latency_p50_s': 4.914,
  'latency_p95_s': 9.846},
 {'model': 'clip-vit-b32',
  'examples': 45,
  'attribute_accuracy_pct': 38.6,
  'image_exact_pct': 0.0,
  'track_exact_pct': 0.0,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 0.742,
  'latency_p95_s': 0.985},
 {'model': 'florence-2-large',
  'examples': 45,
  'attribute_accuracy_pct': 33.0,
  'image_exact_pct': 0.0,
  'track_exact_pct': 0.0,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 1.833,
  'latency_p95_s': 2.678},
 {'model': 'florence-2-large-ft',
  'examples': 45,
  'attribute_accuracy_pct': 31.2,
  'image_exact_pct': 0.0,
  'track_exact_pct': 0.0,
  'structured_valid_pct': 100.0,
  'latency_p50_s': 1.226,
  'latency_p95_s': 2.172}]

In [4]:
field_rows = []
for record in records:
    for field in fields:
        values = []
        for prediction in record["predictions"]:
            if prediction.get("error") is not None:
                continue
            target = (prediction.get("target") or {}).get(field)
            if target in (None, "", "unknown"):
                continue
            actual = (prediction.get("predicted") or {}).get(field)
            values.append((target, actual))
        field_rows.append({
            "model": record["model"],
            "field": field,
            "accuracy_pct": round(100 * sum(bool(actual) and canonical(field, actual) == target for target, actual in values) / len(values), 1),
            "examples": len(values),
        })
field_rows

[{'model': 'qwen3-vl-2b',
  'field': 'gender',
  'accuracy_pct': 37.8,
  'examples': 45},
 {'model': 'qwen3-vl-2b',
  'field': 'age',
  'accuracy_pct': 20.0,
  'examples': 45},
 {'model': 'qwen3-vl-2b',
  'field': 'hair_color',
  'accuracy_pct': 60.0,
  'examples': 15},
 {'model': 'qwen3-vl-2b',
  'field': 'viewpoint',
  'accuracy_pct': 40.0,
  'examples': 45},
 {'model': 'qwen3-vl-2b',
  'field': 'top_color',
  'accuracy_pct': 33.3,
  'examples': 15},
 {'model': 'qwen3-vl-2b',
  'field': 'top_type',
  'accuracy_pct': 66.7,
  'examples': 15},
 {'model': 'qwen3-vl-2b',
  'field': 'bottom_color',
  'accuracy_pct': 100.0,
  'examples': 15},
 {'model': 'qwen3-vl-2b',
  'field': 'bottom_type',
  'accuracy_pct': 26.7,
  'examples': 45},
 {'model': 'qwen3-vl-2b',
  'field': 'accessory',
  'accuracy_pct': 37.8,
  'examples': 45},
 {'model': 'clip-vit-b32',
  'field': 'gender',
  'accuracy_pct': 55.6,
  'examples': 45},
 {'model': 'clip-vit-b32',
  'field': 'age',
  'accuracy_pct': 44.4,
  'exa

## Candidate status

Measured: Qwen3-VL-2B-Instruct, OpenAI CLIP ViT-B/32, Florence-2-large, Florence-2-large-ft.

Not measured: server incumbent Qwen3-VL-8B-Instruct, InternVL3.5-8B, Molmo2-8B, MiniCPM-V-4.5, Gemma family, and GLM family. The local RTX 4050 6GB is not a fair environment for large models, and the remote Jupyter URL redirects to /hub/login without credentials. These candidates must not be described as measured.

## Takeaways

1. Proxy attribute accuracy with missing fields counted as incorrect: Qwen3-VL-2B 39.3%, CLIP ViT-B/32 38.6%, Florence-2-large 33.0%, Florence-2-large-ft 31.2%.
2. All four models had 0% image exact and 0% track exact, so none passed the real track acceptance gate.
3. CLIP was fastest at about 0.74 seconds p50 and is best suited to embedded retrieval or pre-filtering.
4. The next experiment requires reviewed real CCTV track identity labels before selecting a server model or distillation teacher.

## Sources

- [Simuletic CCTV-Pedestrian-1K](https://huggingface.co/datasets/Simuletic/CCTV-Pedestrian-1K-Person-Attribute-Dataset)
- [PA-100K](https://huggingface.co/datasets/tuandunghcmut/PA-100K)
- [Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)
- [OpenAI CLIP ViT-B/32](https://huggingface.co/openai/clip-vit-base-patch32)
- [Florence-2-large](https://huggingface.co/microsoft/Florence-2-large)
- [Florence-2-large-ft](https://huggingface.co/microsoft/Florence-2-large-ft)